# Imports

In [1]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import requests
import yfinance as yf
import time
import os
from tqdm import tqdm

import yfinance as yf
from alpaca.trading.client import TradingClient
from alpaca.data.historical import (
    StockHistoricalDataClient,
    OptionHistoricalDataClient
)
from alpaca.data.requests import (
    StockLatestTradeRequest,
    OptionChainRequest,
    OptionBarsRequest,

)
from alpaca.trading.requests import GetOptionContractsRequest
from alpaca.trading.enums import AssetStatus
from alpaca.data.timeframe import TimeFrame

from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("ALPACA_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET")

In [2]:
stock_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)
option_client = OptionHistoricalDataClient(API_KEY, SECRET_KEY)

# Strategy

### Stock Data
- Get all SP500 and nasdaq stocks ✅✅
- get market cap for each and return only the top 250 ✅✅
-  for loop of all the tickers though the get contracts function to get symbol for all tickers ✅✅
- run symbols from contracts df though the option bars request to get the total volume ✅✅
- Sum the volume by trading day ✅✅
- Get only the last 20 days ✅✅
- Calcualte Average for each ticker ✅✅
- return only the stocks with 10,000 contracts a day ✅✅


### Option Chain
- Get the option chains for filtered stocks that are within 10 dollars of strike pri ce ✅✅
- Filter down to get the strike price closest to the spot price ✅✅
- Filter for 30, 60, 90 DTE ATM contects. +- 5 days for 30 and 60 and +- 8 days for 90 ✅✅
- Pair up front and back contracts contracts  ✅✅
- Get the IVs and the long/short (ask/bid) prices ✅✅

### Signal
- Calculate the forward factor for each DTE pairs ✅✅
- establish ff threshholds for each option combination (short-mid, mid - high)
- Add spread ✅✅
- Recommended FF and Threshold FF ✅✅
- add sugested kelly sizing ✅✅





# Functions

## Utils

In [7]:
def parse_occ_symbol(symbol: str):
    """
    Parse OCC option symbol into components.
    """
    symbol = symbol.strip()
    if symbol[0] == '1':
        symbol = symbol[1:]
        
    underlying = symbol[:-15]
    year = int("20" + symbol[-15:-13])
    month = int(symbol[-13:-11])
    day = int(symbol[-11:-9])
    exp_date =  f"{year}-{month:02d}-{day:02d}"
    option_type = symbol[-9]  # 'C' or 'P'
    strike = int(symbol[-8:]) / 1000.0
    dte = datetime.strptime(exp_date,'%Y-%m-%d') - datetime.today() 
    dte = dte.days
    return {
        "underlying": underlying,
        "expiration": exp_date,
        "dte":dte,
        "type": "call" if option_type == "C" else "put",
        "strike": strike
    }

def get_last_traded(ticker):

    trade_request = StockLatestTradeRequest(symbol_or_symbols=ticker)
    trade = stock_client.get_stock_latest_trade(trade_request)

    return trade[ticker].price


def dte_category(row):
    dte = row

    if dte >=15 and dte<=35:
        return "low"
    elif dte >=45 and dte<=75:
        return 'mid'
    
    elif dte >=80 and dte<=105:
        return 'high'
    
    else:
        return np.nan



def get_next_earnings_dates(tickers):
    """
    Get next earnings date for a list of stock tickers.

    Parameters
    ----------
    tickers : list[str]
        List of ticker symbols (e.g. ["AAPL", "MSFT"])

    Returns
    -------
    pd.DataFrame
        Columns: ticker, earnings_date
    """
    results = []

    for ticker in tqdm(tickers):
        try:
            stock = yf.Ticker(ticker)
            earnings_df = stock.earnings_dates

            if earnings_df is not None and not earnings_df.empty:
                earnings_date = earnings_df.index[0].date()
            else:
                earnings_date = None

        except Exception:
            earnings_date = None

        results.append({
            "underlying": ticker,
            "earnings_date": earnings_date
        })

    return pd.DataFrame(results)


## Stock Selection 

In [8]:
def get_all_tickers():
    sp500_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

    # Use requests to get the content first, then pass to pandas
    response = requests.get(sp500_url, headers=headers)
    sp500 = pd.read_html(response.text)[0]
    sp500_tickers = sp500["Symbol"].tolist()

 

    return sp500_tickers


def filter_tickers(tickers):
    """Returns top 250 stocks by market cap"""
    stocks = yf.Tickers(tickers=tickers)


    data_list = []
    for t in tqdm(tickers, desc="Getting data for all tickers"):
        try:
            # Accessing the info dictionary for each stock
            info = stocks.tickers[t].info
            data_list.append({
                'Ticker': t,
                'Name': info.get('longName'),
                'Market Cap': info.get('marketCap'),
                'Sector': info.get('sector')
            })
        except Exception:
            continue # Skip stocks that fail to load

    # Create DataFrame and sort by Market Cap
    final_df = pd.DataFrame(data_list)
    final_df = final_df.sort_values(by='Market Cap', ascending=False).reset_index(drop=True)[:250]
            

    return final_df


def get_single_ticker_contracts(ticker: str):
    """Returns dictionary with contracts symbol as key and value as another dictionary with contract details"""
    contracts_dic = {}
    today = datetime.today() + timedelta(days=1)
    start = today - timedelta(days=30)


    option_trading_client = TradingClient(API_KEY, SECRET_KEY)

    contracts_inactive_req = GetOptionContractsRequest(
        underlying_symbols=[ticker],
        status=AssetStatus.INACTIVE,
        expiration_date_gte=start.strftime("%Y-%m-%d"),
        limit=10000
        )
    
    contracts_active_req = GetOptionContractsRequest(
        underlying_symbols=[ticker],
        status=AssetStatus.ACTIVE,
        expiration_date_gte=start.strftime("%Y-%m-%d"),
        limit=10000
        )

    active_contracts = option_trading_client.get_option_contracts(contracts_active_req)
    inactive_contracts = option_trading_client.get_option_contracts(contracts_inactive_req)

    contracts = active_contracts.option_contracts + inactive_contracts.option_contracts



    for contract in contracts:
        symbol = contract.symbol
        parsed = parse_occ_symbol(symbol=symbol)
        contracts_dic[symbol] = {
            **parsed,
            'close_date' : contract.close_price_date
        }

    return contracts_dic

def get_tickers_contracts(tickers: list):
    """Return data frame with all option contracts symbols from the input list"""
    contracts_df = pd.DataFrame()
    for ticker in tqdm(tickers, desc="Getting contracts for all tickers"):
        ticker_contract_dic = get_single_ticker_contracts(ticker)
        ticker_contract_df = pd.DataFrame.from_dict(ticker_contract_dic, orient="index").sort_values("expiration").reset_index(names=['symbol'])
        contracts_df = pd.concat([contracts_df, ticker_contract_df])

    return contracts_df

def get_option_bars(symbols):
    """Gets the option bar data for an input list of symbols"""
    bar_data = []
    today = datetime.today()
    start = today - timedelta(days=40)
    batch = 2500
    for i in tqdm(range(0, len(symbols), batch)):
        symbols_batch = symbols[i:i+batch]
        option_bar_request = OptionBarsRequest(
            symbol_or_symbols=symbols_batch,
            timeframe=TimeFrame.Day,
            feed = "indicative",
            start= start.strftime("%Y-%m-%d"),
            end=today.strftime("%Y-%m-%d")
            )
        
        bars = option_client.get_option_bars(option_bar_request)
        bars_list = list(bars.data.values())
        bars_list = [item for sublist in bars_list for item in sublist]
        bar_data += bars_list
        
        

    return bar_data


def daily_option_volume(bar_data):
    """Takes in option bar data dictionary and transforms it into a dataframe with underlying, close date and volume as colums"""

    option_volume_df = pd.DataFrame()
    for bar in tqdm(bar_data):
        symbol = bar.symbol
        volume = bar.volume
        close_date = bar.timestamp
        parsed = parse_occ_symbol(symbol=symbol)
        temp = pd.DataFrame({
                            **parsed,
                            "symbol": [symbol],
                            'close_date': [close_date],
                            'volume': [volume],
                            })
        
        option_volume_df = pd.concat([option_volume_df, temp])
        option_volume_df = option_volume_df.reset_index(drop=True)
        option_volume_df= option_volume_df.groupby(by=['underlying', 'close_date']).sum()['volume'].to_frame().reset_index()

    return option_volume_df


def averge_daily_volume(daily_option_volume_df, last_n_days):
    daily_option_volume_df['date_rank'] = daily_option_volume_df.groupby(by='underlying')['close_date'].rank(method='min', ascending=False)
    avg_option_volume_df = daily_option_volume_df[daily_option_volume_df['date_rank'] <= last_n_days].groupby(by='underlying')['volume'].mean().to_frame().reset_index()
    avg_option_volume_df = avg_option_volume_df.sort_values('volume', ascending=False).reset_index(drop=True)

    return avg_option_volume_df

## Option Chain Processing

In [9]:
def get_chains(ticker, dtes, dte_buffer, strike_buffer):

    chains = {}
    today = datetime.today()
    for dte in dtes:
        target_dte = today + timedelta(days=dte)
        lower_dte = target_dte - timedelta(days=dte_buffer)
        upper_dte = target_dte + timedelta(days=dte_buffer)
        latest_price = get_last_traded(ticker)

        chain_request = OptionChainRequest(
        underlying_symbol=ticker,
        expiration_date_gte=lower_dte.strftime("%Y-%m-%d"),
        expiration_date_lte=upper_dte.strftime("%Y-%m-%d"),
        strike_price_gte=latest_price - strike_buffer,
        strike_price_lte= latest_price + strike_buffer,
        option_type="call"
        )

        chain = option_client.get_option_chain(chain_request)

        for symbol, data in chain.items():
            parsed = parse_occ_symbol(symbol=symbol)
            if parsed['type'] == 'call':
                chains[symbol] =  {
                    **parsed,
                    # **dict(data.greeks),
                    "iv": data.implied_volatility,
                    "bid": data.latest_quote.bid_price,
                    "ask": data.latest_quote.ask_price,

                }

    return chains

def chain_dic_to_df(chains, ticker):        
    chains_df = pd.DataFrame.from_dict(chains, orient="index").sort_values("strike")
    chains_df = chains_df.reset_index(names='symbol')
    chains_df['dte_category'] = chains_df['dte'].apply(dte_category)
    chains_df = chains_df.dropna(subset='dte_category')
    chains_df['spread'] = chains_df['ask'] - chains_df['bid']

    latest_price = get_last_traded(ticker)
    chains_df['strike-latest'] = abs(chains_df['strike'] - latest_price)
    chains_df['latest_price'] = latest_price
    chains_df = chains_df[chains_df['strike-latest'].min() ==  chains_df['strike-latest']] # keeping the strikes that are the closest strike price


    keep_columns = ['underlying', 'latest_price','expiration','dte', 'dte_category','strike','iv','bid','ask', 'spread']
    chains_df= chains_df[keep_columns]

    return chains_df



### Signal Processing

In [10]:
def pair_contracts(chains_df):
    low_dte_df = chains_df[chains_df['dte_category'] == 'low']
    mid_dte_df = chains_df[chains_df['dte_category'] == 'mid']
    high_dte_df = chains_df[chains_df['dte_category'] == 'high']

    low_mid_df = pd.merge(low_dte_df, mid_dte_df, how='cross', suffixes=("_s","_l"))
    mid_high_df = pd.merge(mid_dte_df, high_dte_df, how='cross', suffixes=("_s","_l"))
    low_high_df = pd.merge(low_dte_df, high_dte_df, how='cross', suffixes=("_s","_l"))
    strat_df = pd.concat([low_mid_df, mid_high_df, low_high_df])

    return strat_df

def forward_factor(row):
    t1 = row['short_dte']/365
    t2 = row['long_dte']/365
    v1 = row['short_iv']
    v2 = row['long_iv']
    ff_var = (v2**2*t2 - v1**2*t1)/(t2-t1)
    ff_var = np.sqrt(ff_var)
    return ((v1 - ff_var) / ff_var)*100


def process_strat_df(strat_df, chains_df):

    new_cols = [f"short_{col}" for col in chains_df.columns] + [f"long_{col}" for col in chains_df.columns]
    new_col_mapping = dict(zip(strat_df.columns, new_cols))
    strat_df = strat_df.rename(columns=new_col_mapping)
    strat_df = strat_df.drop(columns=['long_underlying','long_strike','short_ask','long_bid', 'long_latest_price'])
    strat_df = strat_df.rename(columns={
        'short_underlying': 'underlying',
        'short_strike':'strike',
        'short_latest_price': 'latest_price'
    })

    strat_df = strat_df.reset_index(drop=True)
    strat_df['ff_factor']  = strat_df.apply(forward_factor, axis=1)
    strat_df['total_price'] = strat_df['long_ask'] - strat_df['short_bid']
    reorder_cols = ['underlying', 'latest_price','strike',  'short_expiration', 'long_expiration', 'short_dte_category','long_dte_category', 
                    'short_spread','long_spread', 'short_bid','long_ask', 'total_price', 'short_dte', 'long_dte', 'short_iv','long_iv', 'ff_factor']

    strat_df = strat_df[reorder_cols]

    return strat_df

### FF Threshold Logic
- low - mid : 23 % : Increases the FF to limit the number of entry signals
- low - high : 23 % : Increases the FF to limit the number of entry signals
- mid - high : 41.46% : This is the FF cross over thresh hold. its about 4 trades a month and does not require me to reduce the the FF to get the 20 trades


With this combination, I should average about 20 + 20 + 4 = 44 trades a month. Which is very managable.  The increase FF on the low -mid and the low - high means a higher win rate and average return per trade
Below is the mininim FF Threshold required to make a profit according to historical data
- low - mid : 23 % 
- low - high : 23 % 
- mid - high : 41.46% 


### Kelly Criterion Logic
- low - mid : 8 % Half
- low - high : 5 %  Quarter
- mid - high : 4.5 Half

- Tried to keep all sizing between 2-8% and maximize CAGR and Sharpe Ratio



In [11]:
ff_recommended_dic = {
    'short_dte_category':['low','low','mid'],
    'long_dte_category': ['mid','high','high'],
    'recommended_ff': [23, 23 , 41.5]
}


ff_threshold_dic = {
    'short_dte_category':['low','low','mid'],
    'long_dte_category': ['mid','high','high'],
    'threshold_ff': [14.1, 13.23 , 41.5]
}

kelly_dic = {
    'short_dte_category':['low','low','mid'],
    'long_dte_category': ['mid','high','high'],
    'kelly': [0.08, 0.05, 0.045]
}

ff_recommended_df = pd.DataFrame.from_dict(ff_recommended_dic)
ff_threshold_df = pd.DataFrame.from_dict(ff_threshold_dic)
kelly_df = pd.DataFrame.from_dict(kelly_dic)

ff_recommended_df

,short_dte_category,long_dte_category,recommended_ff
0,low,mid,23.0
1,low,high,23.0
2,mid,high,41.5


# Parameters

In [12]:
DTES = [30,60,90]
DTE_BUFFER = 15
STRIKE_BUFFER = 5
LAST_N_DAYS_AVERAGE_VOLUME = 20
PORTFOLIO_SIZE = 40000

# Script

## Stock Selection

In [13]:
tickers = get_all_tickers()   # gets sp500 tickers
tickers_df = filter_tickers(tickers) # gets market cap for all tickers

C:\Users\jacob\AppData\Local\Temp\ipykernel_50440\1815124572.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500 = pd.read_html(response.text)[0]
Getting data for all tickers: 100%|██████████| 503/503 [02:04<00:00,  4.03it/s]


In [ ]:
tickers = tickers_df['Ticker'].tolist() # list of tickers
tickers_contracts_df = get_tickers_contracts(tickers) # gets all the symbols in a df of all the tickers
tickers_contracts_df = tickers_contracts_df.dropna(subset='close_date') # If it has a none then there was no volume for those option contract in that period

Getting contracts for all tickers:   3%|▎         | 8/250 [00:08<04:34,  1.13s/it]

In [ ]:
symbols = tickers_contracts_df['symbol'].tolist() # list of all the symbols
bar_data = get_option_bars(symbols)  # dictionary of option bar data of all the relevant tickers
daily_option_volume_df = daily_option_volume(bar_data) # gets the daily volume by ticker

In [9]:
daily_option_volume_file_path = 'data/1.2.25_daily_option_volume.csv'
# daily_option_volume_df.to_csv(daily_option_volume_file_path, index=False)

In [10]:
daily_option_volume_df = pd.read_csv(daily_option_volume_file_path)
averge_daily_volume_df = averge_daily_volume(daily_option_volume_df, last_n_days=LAST_N_DAYS_AVERAGE_VOLUME) # filters tickers by for those that have more than 10,000 average daily volume in the last 20 days

In [11]:
eligible_tickers = averge_daily_volume_df[averge_daily_volume_df['volume']>9500]['underlying'].tolist()

## Option Chain and Signal Processing 

In [12]:
tickers =  ["AAPL","TSLA","NVDA"]
ticker = tickers[0]

In [13]:
strats_df = pd.DataFrame()

for ticker in tqdm(eligible_tickers):
    chains = get_chains(ticker, DTES, DTE_BUFFER, STRIKE_BUFFER)
    if len(chains) == 0:
        continue
    chains_df = chain_dic_to_df(chains, ticker)
    strat_df = pair_contracts(chains_df)
    strat_df = process_strat_df(strat_df, chains_df)
    strats_df = pd.concat([strats_df, strat_df])


100%|██████████| 105/105 [06:02<00:00,  3.46s/it]


In [14]:
strats_df = strats_df.reset_index(drop=True)
strats_df = pd.merge(strats_df, ff_recommended_df, on=('short_dte_category','long_dte_category'))
strats_df = pd.merge(strats_df, ff_threshold_df, on=('short_dte_category','long_dte_category'))
strats_df = pd.merge(strats_df, kelly_df, on=('short_dte_category','long_dte_category'))
strats_df = strats_df[strats_df['ff_factor']>= strats_df['recommended_ff']]
strats_df = strats_df.sort_values(by='ff_factor', ascending=False).reset_index(drop=True)
strats_df['trade_size'] = strats_df['kelly'] * PORTFOLIO_SIZE
strats_df['total_price'] = strats_df['total_price']* 100
strats_df['n_contracts'] = (strats_df['trade_size'] / strats_df['total_price'])
strats_df['total_spread'] = strats_df['short_spread'] +  strats_df['long_spread']
strats_df = strats_df[((strats_df['short_bid'] + strats_df['long_ask']) *.10)<= strats_df['total_spread']]   # only want trades that have a total spread less thatn 10 percent the price of all the options



In [15]:
earnings_df = get_next_earnings_dates(strats_df['underlying'].unique())
strats_df_merge = strats_df.merge(earnings_df)

100%|██████████| 8/8 [00:05<00:00,  1.55it/s]


In [16]:
pd.set_option('display.max_columns', None)
strats_df_merge[abs((pd.to_datetime(strats_df_merge['short_expiration']) -  pd.to_datetime(strats_df_merge['earnings_date'])).dt.days) >= 7]

,underlying,latest_price,strike,short_expiration,long_expiration,short_dte_category,long_dte_category,short_spread,long_spread,short_bid,long_ask,total_price,short_dte,long_dte,short_iv,long_iv,ff_factor,recommended_ff,threshold_ff,kelly,trade_size,n_contracts,total_spread,earnings_date
0,V,353.81,355.0,2026-02-06,2026-02-20,low,mid,1.760,0.65,9.05,10.71,166.0,31,45,0.2475,0.2098,223.832808,23.0,14.1,0.08,3200.0,19.277108,2.410,2026-01-27
1,KMI,27.92,28.0,2026-01-30,2026-02-20,low,mid,0.060,0.07,0.57,0.73,16.0,24,45,0.2157,0.1745,96.274257,23.0,14.1,0.08,3200.0,200.000000,0.130,2026-01-21
4,XOM,125.40,125.0,2026-02-06,2026-02-20,low,mid,0.880,0.25,3.13,4.08,95.0,31,45,0.2093,0.1904,49.787950,23.0,14.1,0.08,3200.0,33.684211,1.130,2026-01-30
5,VZ,40.23,40.0,2026-02-06,2026-02-20,low,mid,0.400,0.09,0.74,1.09,35.0,31,45,0.1609,0.1469,46.650079,23.0,14.1,0.08,3200.0,91.428571,0.490,2026-01-30
6,IBM,294.95,295.0,2026-02-06,2026-02-20,low,mid,2.720,0.29,11.02,14.24,322.0,31,45,0.3481,0.3256,29.325105,23.0,14.1,0.08,3200.0,9.937888,3.010,2026-01-28
9,USB,55.50,55.0,2026-01-30,2026-02-20,low,mid,0.269,0.22,1.76,2.46,70.0,24,45,0.2770,0.2538,23.458310,23.0,14.1,0.08,3200.0,45.714286,0.489,2026-01-20
10,IBM,294.95,295.0,2026-02-06,2026-03-20,low,mid,2.720,1.61,11.02,18.25,723.0,31,73,0.3481,0.3120,23.260454,23.0,14.1,0.08,3200.0,4.426003,4.330,2026-01-28
11,XOM,125.40,125.0,2026-02-06,2026-03-20,low,mid,0.880,0.21,3.13,5.09,196.0,31,73,0.2093,0.1877,23.113439,23.0,14.1,0.08,3200.0,16.326531,1.090,2026-01-30


In [141]:
ff_recommended_df

,short_dte_category,long_dte_category,recommended_ff
0,low,mid,23.0
1,low,high,23.0
2,mid,high,41.5


In [142]:
ff_threshold_df

,short_dte_category,long_dte_category,threshold_ff
0,low,mid,14.10
1,low,high,13.23
2,mid,high,41.50


In [143]:
kelly_df

,short_dte_category,long_dte_category,kelly
0,low,mid,8.0
1,low,high,5.0
2,mid,high,4.5
